# A1 – Opisna statistika

**Sklop A: ena spremenljivka · vodena demonstracija · Olist**

Ta notebook spremlja razdelek **A. Ena spremenljivka**

Po koncu znamo:
- povzeti sredino, razpršenost in obliko numeričnega stolpca;
- razložiti histogram, škatlo z brki in meje po pravilu 1,5 · IQR;
- izbrati statistiko glede na pomen spremenljivke;
- pregledati kategorije, redke vrednosti in manjkajoče podatke.

Pri vsakem izpisu se vprašamo: **Kaj predstavlja en zapis? Koliko vrednosti poznamo?
Kaj lahko iz izpisa sklepamo?**


## Priprava

Notebook je samostojen in deluje lokalno ter v Google Colabu. Celice izvajaj od
zgoraj navzdol s **Shift + Enter**. Ob menjavi stolpcev ponovi ustrezne naslednje celice.

Prva celica poišče `utils/orodja.py`; v Colabu po potrebi prenese repozitorij in namesti
knjižnice iz `requirements.txt`. Lokalno jih enkrat namestimo po navodilih v README.
Druga celica pokliče skupni `poskrbi_za_podatke()` in vrne mapo `DATA`.
Funkcija uporabi pripravljene ali že prenesene podatke; manjkajoče prenese s Kaggla.
CSV-je nato beremo iz `DATA`, ne glede na mapo, v kateri je zvezek odprt.
Izvornih vrednosti ne popravljamo in rezultatov vaj ne zapisujemo samodejno v datoteke.


In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)

### Skupni podatkovni vhod
`poskrbi_za_podatke()` iz `utils/orodja.py` pripravi skupno mapo podatkov in jo vrne kot `DATA`.


In [ ]:
from utils.orodja import poskrbi_za_podatke

DATA = poskrbi_za_podatke()

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 15)
pd.set_option("display.max_colwidth", 35)
plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11})

### Podatki in enota opazovanja

Za vrednost naročila seštejemo **cene izdelkov brez poštnine** po `order_id`.
Posamezna vrstica v datoteki izdelkov naročila ni celotno naročilo.
Vrednost, poštnino in število izdelkov nato opisujemo za **dostavljena naročila**.

Za kategorije dodamo lokacijo kupca. `customer_id` poveže naročilo z lokacijo;
`customer_unique_id` povezuje nakupe istega kupca. Identifikatorji niso meritve.

Ocene obravnavamo **ločeno, po zapisih ocen za dostavljena naročila**. Eno naročilo
ima lahko več zapisov ocen. Pri teh frekvencah je imenovalec število znanih ocen,
ne število naročil. S tem se izognemo podvajanju denarnih vrednosti pri združevanju.


In [ ]:
mapa_podatkov = DATA / "trgovina"
narocila = pd.read_csv(mapa_podatkov / "olist_orders_dataset.csv")
izdelki_narocil = pd.read_csv(mapa_podatkov / "olist_order_items_dataset.csv")
kupci = pd.read_csv(
    mapa_podatkov / "olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "string"},
)
vse_ocene = pd.read_csv(mapa_podatkov / "olist_order_reviews_dataset.csv")

display(pd.DataFrame({
    "tabela": ["naročila", "izdelki naročil", "kupci", "zapisi ocen"],
    "vrstice": [len(narocila), len(izdelki_narocil), len(kupci), len(vse_ocene)],
}))

In [ ]:
# Postavke naročil najprej povzamemo na eno vrstico na naročilo.
zneski_narocil = izdelki_narocil.groupby("order_id", as_index=False).agg(
    vrednost_narocila=("price", "sum"),
    postnina=("freight_value", "sum"),
    stevilo_izdelkov=("order_item_id", "size"),
    stevilo_prodajalcev=("seller_id", "nunique"),
)

df = (
    narocila.loc[narocila["order_status"].eq("delivered")]
    .merge(zneski_narocil, on="order_id", how="left", validate="one_to_one")
    .merge(
        kupci[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
        on="customer_id", how="left", validate="one_to_one",
    )
    .copy()
)
df["dodatni_izdelki"] = df["stevilo_izdelkov"] - 1

# Ocene ostanejo svoja tabela. Več ocen ne pomnoži vrednosti naročil v df.
ocene = vse_ocene.loc[vse_ocene["order_id"].isin(df["order_id"])].copy()

print(f"Dostavljena naročila: {len(df):,}")
print(f"Zapisi ocen teh naročil: {len(ocene):,}")
print(f"Različna naročila z oceno: {ocene['order_id'].nunique():,}")
display(df[["vrednost_narocila", "postnina", "stevilo_izdelkov", "customer_state"]].head())

## 1. Prvi pregled: kaj povzame `describe()`?

Izberemo nekaj smiselnih numeričnih stolpcev. `count` je število **znanih** vrednosti,
ne nujno število vseh vrstic. `.T` obrne tabelo, da je vsak stolpec v svoji vrstici.

**Pred izpisom:** ali pričakujemo, da bo povprečna vrednost naročila podobna mediani?


In [ ]:
numericni_stolpci = [
    "vrednost_narocila", "postnina", "stevilo_izdelkov", "stevilo_prodajalcev",
]
display(df[numericni_stolpci].dtypes.rename("tip_v_pandasu").to_frame())
display(df[numericni_stolpci].describe().T.round(2))

## 2. Mere središčnosti: povprečje, mediana, modus

Povprečje je občutljivo na velike vrednosti. Mediana je proti ekstremom odpornejša.
Modus je najpogostejša vrednost; možnih je tudi več modusov.

Pri vrednosti naročila uporabljamo izvorne denarne enote **BRL**.


In [ ]:
s_vrednost = df["vrednost_narocila"]
display(s_vrednost.agg(["count", "mean", "median", "min", "max"]).round(2))

modusi = s_vrednost.mode()
print(f"Število modusov: {len(modusi)}")
display(modusi.head(10).rename("modus_vrednosti_narocila_BRL").to_frame())

**Vprašanje:** kateri podatek bi uporabili za opis običajnega naročila in
katerega za načrtovanje skupnega prometa?


## 3. Mere razpršenosti: razpon, varianca, standardni odklon, kvartili

Razpon določata najmanjša in največja vrednost. Standardni odklon je v BRL,
varianca v BRL². IQR opiše širino srednjih 50 % vrednosti.

Pandasova `.var()` in `.std()` privzeto uporabita **`ddof=1`**: pri vzorčni varianci
delimo z n − 1. Če opisujemo celotno populacijo in želimo deljenje z n, uporabimo
`ddof=0`. Izbiro pri primerjavi izpisov ohranimo enako.


In [ ]:
znane_vrednosti = df["vrednost_narocila"].dropna()
q1, q2, q3 = znane_vrednosti.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1

razprsenost = pd.Series({
    "minimum": znane_vrednosti.min(),
    "maksimum": znane_vrednosti.max(),
    "razpon": znane_vrednosti.max() - znane_vrednosti.min(),
    "varianca (ddof=1; BRL²)": znane_vrednosti.var(ddof=1),
    "standardni odklon (ddof=1; BRL)": znane_vrednosti.std(ddof=1),
    "Q1": q1,
    "Q2 / mediana": q2,
    "Q3": q3,
    "IQR": iqr,
})
display(razprsenost.round(2).to_frame("vrednost"))

## 4. Oblika porazdelitve: histogram, rep in ničle

Prvi histogram vsebuje vse znane vrednosti. Drugi prikaže le območje do 99. percentila,
da lahko vidimo glavnino podatkov. Omejitev velja **le za prikaz**, ne za tabelo ali
prejšnje statistike. Število razredov vpliva na videz histograma.


In [ ]:
meja_prikaza = znane_vrednosti.quantile(0.99)
glavnina = znane_vrednosti.loc[znane_vrednosti.le(meja_prikaza)]

fig, osi = plt.subplots(1, 2, figsize=(11, 4))
osi[0].hist(znane_vrednosti, bins=50, edgecolor="white")
osi[0].set_title("Vsa naročila: tudi dolg desni rep")
osi[1].hist(glavnina, bins=40, edgecolor="white")
osi[1].set_title("Prikaz do 99. percentila")
for os in osi:
    os.set_xlabel("Vrednost naročila (BRL)")
    os.set_ylabel("Število naročil")
fig.tight_layout()
plt.show()

### Kaj pomeni veliko ničel?

Izpeljani stolpec `dodatni_izdelki = stevilo_izdelkov − 1` je pri naročilu z enim
izdelkom enak 0. Ločimo vprašanji: **ali naročilo vsebuje dodatne izdelke** in
**koliko dodatnih izdelkov vsebuje, kadar jih ima**. Ničla ima tukaj jasen pomen.
Deleže računamo med znanimi vrednostmi.


In [ ]:
dodatni = df["dodatni_izdelki"].dropna()
pozitivni = dodatni.loc[dodatni.gt(0)]

display(pd.Series({
    "znanih vrednosti": dodatni.size,
    "delež ničel (%)": dodatni.eq(0).mean() * 100,
    "povprečje med vsemi znanimi": dodatni.mean(),
    "mediana med vsemi znanimi": dodatni.median(),
    "povprečje samo med pozitivnimi": pozitivni.mean(),
}).round(2).to_frame("vrednost"))

fig, os = plt.subplots()
dodatni.value_counts().sort_index().plot.bar(ax=os)
os.set(xlabel="Število dodatnih izdelkov", ylabel="Število naročil",
       title="Nič pomeni naročilo z enim izdelkom")
fig.tight_layout()
plt.show()

## 5. Pomen spremenljivke določa statistiko

| Stolpec | Pomen | Smiselni povzetki |
|---|---|---|
| `vrednost_narocila` | Numerična denarna vrednost | Povprečje, mediana, odklon, kvartili |
| `stevilo_izdelkov` | Diskretno število | Frekvence, mediana, tudi povprečje glede na vprašanje |
| `review_score` | Ordinalna ocena 1–5 | Frekvence, mediana, modus; povprečje s pojasnjeno predpostavko |
| `customer_state` | Nominalna kategorija: brazilska zvezna država | Frekvence, deleži, modus |
| `order_id` | Identifikator | Število različnih ID-jev; povprečje nima poslovnega pomena |

Ocene spodaj štejejo **zapise ocen za dostavljena naročila**, ne po ene ocene na
naročilo. Urejene številčne oznake še ne zagotavljajo enakih razmikov med ocenami.


In [ ]:
display(df["stevilo_izdelkov"].value_counts().sort_index().rename("narocila").to_frame())

ocene_znane = ocene["review_score"].dropna()
frekvence_ocen = pd.DataFrame({
    "število ocen": ocene_znane.value_counts(),
    "delež (%)": ocene_znane.value_counts(normalize=True) * 100,
}).sort_index()
display(frekvence_ocen.round(2))
print(f"Mediana ocen: {ocene_znane.median():.1f}; modus: {ocene_znane.mode().tolist()}")

## 6. Kategorije: frekvence, kardinalnost in redke vrednosti

Oznaka brazilske zvezne države (npr. SP ali RJ) je kategorija, ne številska meritev.
Za primerjavo izpišemo števila
in deleže. Pri mestu je kategorij več. Meja 100 zapisov spodaj je izbran prag za
pregled, ne univerzalna definicija redke kategorije.


In [ ]:
kategorija = df["customer_state"]
stevila = kategorija.value_counts(dropna=False)
delezi = kategorija.value_counts(dropna=False, normalize=True) * 100
frekvence = pd.DataFrame({"naročila": stevila, "delež (%)": delezi})

print(f"Različnih znanih zveznih držav: {kategorija.nunique()}")
print(f"Različnih znanih mest: {df['customer_city'].nunique()}")
display(frekvence.round(2))

prag_redkosti = 100
redke_kategorije = stevila.loc[stevila.lt(prag_redkosti)]
print(f"Kategorij z manj kot {prag_redkosti} naročili: {len(redke_kategorije)}")
display(redke_kategorije.rename("naročila v redki kategoriji").to_frame())

In [ ]:
# Manjkajočo kategorijo poimenujemo samo za prikaz; izvornih podatkov ne popravljamo.
prikaz = df["customer_state"].fillna("MANJKA").value_counts()
fig, os = plt.subplots(figsize=(10, 4))
prikaz.plot.bar(ax=os)
os.set(xlabel="Zvezna država kupca (oznaka)", ylabel="Število naročil",
       title="Nominalna spremenljivka: frekvenčni prikaz")
fig.tight_layout()
plt.show()

## 7. Manjkajoče vrednosti so del opisa

`isna()` zazna manjkajoče vrednosti, ki jih je Pandas prepoznal. Posebnih kod ne
moremo razglasiti za manjkajoče brez poznavanja njihovega pomena.

Spodaj sta **dva ločena pregleda**: vsa naročila in zapisi ocen dostavljenih naročil.
Manjkajoč datum dostave in neizpolnjen naslov ocene imata različna možna pomena.
Niti enega ne nadomestimo samodejno z ničlo. Naslov ocene je neobvezno besedilo;
njegova odsotnost še ne pomeni, da je treba izbrisati oceno.


In [ ]:
for ime, tabela in {"Vsa naročila": narocila, "Ocene dostavljenih naročil": ocene}.items():
    print()
    print(f"{ime}: {len(tabela):,} zapisov")
    pregled = pd.DataFrame({
        "znanih": tabela.count(),
        "manjkajočih": tabela.isna().sum(),
        "manjka (%)": tabela.isna().mean() * 100,
    }).sort_values("manjka (%)", ascending=False)
    display(pregled.round(2))
    print(f"Brez vseh manjkajočih vrednosti bi ostalo {tabela.notna().all(axis=1).sum():,} zapisov.")